CURSO: Machine Learning en Geociencias
Profesor: Edier Aristizábal (evaristizabalg@unal.edu.co)
Credits: El contenido de este notebook proviene de varias fuentes, such  [Sunghyun Ahn](https://medium.com/data-science-collective/understanding-na%C3%AFve-bayes-for-classification-ml-66ae678340b2) en Medium. Se ha hecho todo lo posible por identificar a los titulares de los derechos de los materiales usados en este notebook. The author apolozgies for any unintentional omissions and would be pleased to add an acknowledgment in future editions.

# 26. Bayes ingenuo gaussiano para clasificación

Es un algoritmo de clasificación probabilística basado en el Teorema de Bayes, con la suposición de que las características son condicionalmente independientes dado el valor de la clase. Esta suposición simplificadora permite al modelo calcular la probabilidad de cada característica por separado y luego combinarlas para determinar la probabilidad total de cada clase.

La parte “Naive” del nombre proviene de la suposición simplificadora de que todas las características son independientes entre sí, dado el valor de la clase. En escenarios del mundo real, las características (como las palabras en un correo electrónico o los síntomas en un paciente) a menudo presentan cierto grado de dependencia entre ellas, pero ignorar estas dependencias generalmente no afecta el rendimiento tanto como se podría esperar.

Este concepto está en el corazón del Teorema de Bayes, el cual es fundamental para comprender el algoritmo de Naïve Bayes. Antes de continuar, presentemos el Teorema de Bayes, ya que lo referiremos a lo largo de este artículo.

$$
P(H \mid E) = \frac{P(E \mid H) \cdot P(H)}{P(E)}
$$

- **P(E ∣ H)**: Probabilidad de observar la evidencia *E* si la hipótesis *H* es verdadera (*verosimilitud*).
- **P(H)**: Probabilidad previa de la hipótesis *H* antes de observar la evidencia (*apriori*).
- **P(E)**: Probabilidad total de la evidencia *E* considerando todas las hipótesis posibles (*marginal*).



#### Ejemplo

Conozcamos a Stewart, un jefe en una planta de manufactura que produce teléfonos inteligentes para Samsung. Su empresa quiere predecir si un teléfono recién ensamblado será defectuoso o no defectuoso, basándose en tres características recolectadas en la línea de producción:

- **Puntaje del sensor (Sensor Score)**
- **Línea de ensamblaje (Assembly Line)**
- **Experiencia del trabajador (Worker Experience)**

Cada teléfono pasa por un control de calidad, y Stewart dispone de datos de 100 teléfonos, incluyendo su estado (defectuoso o no defectuoso).

Ahora, supongamos que queremos clasificar un nuevo teléfono con los siguientes atributos:

- **Puntaje del sensor** = 65  
- **Línea de ensamblaje** = "Línea B"  
- **Experiencia del trabajador** = "Media"

### ¿Cuál es la probabilidad de que un teléfono sea defectuoso, sin considerar sus características?

Para calcular la **probabilidad apriori** de que un teléfono sea defectuoso, independientemente de sus características, utilizamos la siguiente fórmula:

$$
P(\text{Defectuoso}) = \frac{\text{Número de teléfonos defectuosos}}{\text{Número total de teléfonos}}
$$

Si Stewart tiene datos de **100 teléfonos** y **20 de ellos** son defectuosos, entonces:

$$
P(\text{Defectuoso}) = \frac{20}{100} = 0{,}2
$$

Esto significa que hay una **probabilidad del 20%** de que un teléfono seleccionado al azar sea defectuoso, **antes** de considerar características como el puntaje del sensor, la línea de ensamblaje o la experiencia del trabajador.

Ahora que hemos establecido la probabilidad previa de que un teléfono sea defectuoso, ¡es momento de incorporar las características en nuestro análisis!

En este punto, tenemos todos estos datos valiosos sobre cómo las características se correlacionan con el estado de defectuoso. La pregunta clave que queremos responder es:

*¿Cuánto afecta cada característica a la probabilidad de que un teléfono sea defectuoso?*

En términos matemáticos, necesitamos calcular \( P(\text{Característica} \mid \text{Defectuoso}) \). Esto es, la probabilidad de observar una determinada característica dado que el teléfono es defectuoso.

Esto se llama **probabilidad condicional**.

En estadística, hablamos de **probabilidad condicional** porque existe una condición previa dada (un teléfono defectuoso) antes de calcular algo que nos interesa (una característica).

En nuestro caso, tenemos tres características (**Puntaje del Sensor**, **Línea de Ensamblaje** y **Experiencia del Trabajador**), por lo que debemos determinar tres probabilidades condicionales separadas:

- \( P(\text{Puntaje del Sensor} \mid \text{Defectuoso}) \)
- \( P(\text{Línea de Ensamblaje} \mid \text{Defectuoso}) \)
- \( P(\text{Experiencia del Trabajador} \mid \text{Defectuoso}) \)

Recuerda que estamos tratando de estimar para un nuevo teléfono que tiene:

- Un **puntaje del sensor** de "65",
- Fabricado en la **línea de ensamblaje B**,
- Y ensamblado por un trabajador con **experiencia de nivel medio**.

Es bastante sencillo pensar en esto de manera intuitiva al calcular estas probabilidades condicionales (cuando se trata de datos categóricos como la Línea de Ensamblaje y la Experiencia del Trabajador).

Al observar nuevamente los datos, podemos ver que, de la **línea de ensamblaje B**, hubo aproximadamente **10 teléfonos defectuosos** y **11 no defectuosos**. Entonces, ¿qué podemos decir sobre \( P(\text{característica} \mid \text{defectuoso}) \)? Sabemos que hay **20 teléfonos defectuosos** en todo el conjunto de datos de 100 teléfonos. De esos 20 defectuosos, **10 provienen de la línea B**. ¡Así que el cálculo se vuelve muy simple!

> **20 teléfonos defectuosos en total, 10 de ellos de la Línea B:**
>
> $$
> P(\text{Línea B} \mid \text{Defectuoso}) = \frac{10}{20} = 0{,}5
> $$

Podemos hacer lo mismo con los teléfonos **no defectuosos**:

> **80 teléfonos no defectuosos en total, 11 de ellos de la Línea B:**
>
> $$
> P(\text{Línea B} \mid \text{No Defectuoso}) = \frac{11}{80} \approx 0{,}1375
> $$

De manera similar, podemos calcular las probabilidades para la **experiencia del trabajador**, ya que sigue la misma lógica categórica.

Con estas dos probabilidades condicionales calculadas, pasaremos a la siguiente característica (**puntaje del sensor**), que es ligeramente diferente a las anteriores.

Para características numéricas como el **Puntaje del Sensor**, no podemos simplemente contar ocurrencias, ya que estos valores son continuos y no discretos. Al ser continuos, pueden tomar infinitos valores posibles dentro de un rango determinado.

Si has seguido mis artículos, probablemente me habrás visto destacar la diferencia entre **distribuciones discretas y continuas**.

Aquí es donde esa distinción se vuelve importante.

Dado que los puntajes del sensor pueden tomar cualquier valor dentro de un rango, necesitamos un enfoque diferente para estimar las probabilidades. Vamos a revisar primero un enfoque más intuitivo y luego uno más matemático.

Antes de pasar a métodos más avanzados, pensemos primero en una solución intuitiva. Considera la siguiente distribución de teléfonos defectuosos y no defectuosos según su puntaje del sensor:

- De los **20 teléfonos defectuosos**, **8 tienen un puntaje de sensor entre 60 y 70**.
- De los **80 teléfonos no defectuosos**, **solo 10 están en ese mismo rango de puntaje**.

Dado que estamos tratando de clasificar un teléfono nuevo con un **puntaje de sensor de 65**, podemos estimar su **probabilidad condicional** con base en ese rango:

> **Probabilidad condicional para puntaje del sensor usando binning:**
>
> $$
> P(60 \leq \text{Sensor Score} \leq 70 \mid \text{Defectuoso}) = \frac{8}{20} = 0{,}4
> $$

Este método se conoce como **binning**, donde dividimos el puntaje del sensor en rangos y los tratamos como categorías. Aunque puede funcionar para conjuntos de datos pequeños, tiene limitaciones.

Por ejemplo... ¿Qué pasa si el nuevo teléfono tiene un puntaje de sensor de **65.3**? Nuestro enfoque de binning no maneja bien valores continuos.

Afortunadamente, existe un mejor enfoque para tratar este problema.


### Un enfoque más efectivo: **Naïve Bayes Gaussiano**

En lugar de usar binning, una forma más efectiva de manejar características numéricas continuas es asumir que siguen una distribución de probabilidad, en particular, una **distribución Gaussiana (Normal)**.

Usando la **función de densidad de probabilidad (PDF)** de una distribución normal, podemos estimar:

$$
P(\text{Sensor Score} = x \mid \text{Defectuoso}) = \frac{1}{\sqrt{2\pi\sigma^2}} \cdot \exp\left( -\frac{(x - \mu)^2}{2\sigma^2} \right)
$$

donde:

- \( \mu \): Media de los puntajes del sensor en teléfonos defectuosos  
- \( \sigma \): Desviación estándar de esos puntajes  
- \( x \): Puntaje del sensor del nuevo teléfono

Con Python o R, podemos calcular fácilmente la media y la desviación estándar de los teléfonos defectuosos para aplicar esta fórmula.



In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv("your_file.csv")

# Compute mean and standard deviation
mean_value = df["Sensor Score"].mean()
std_value = df["Sensor Score"].std()

print(f"Mean (μ): {mean_value}")
print(f"Standard Deviation (σ): {std_value}")

Supongamos que, utilizando el siguiente código, tenemos los valores para los teléfonos defectuosos (\( \mu = 59 \) y \( \sigma = 5 \)). Sabemos que el nuevo teléfono tiene un **Puntaje del Sensor = 65**. Al reemplazar estos valores en la fórmula de la distribución Gaussiana, obtenemos:

$$
P(\text{Sensor Score} = 65 \mid \text{Defectuoso}) = 0{,}4
$$


Este valor es, esencialmente, el mismo que la probabilidad que obtuvimos anteriormente utilizando el enfoque de **binning**.

Ahora hemos calculado todas las **probabilidades condicionales** tanto para los teléfonos defectuosos como para los no defectuosos, en las tres características analizadas.

¡Ahora que tenemos todas nuestras probabilidades condicionales, es momento de aplicar el algoritmo de **Naïve Bayes**!

En el corazón de la clasificación **Naïve Bayes** se encuentra el **Teorema de Bayes** (como hemos venido explicando desde el inicio de este artículo). Este teorema nos permite determinar la probabilidad de una hipótesis \( H \) (por ejemplo, que un teléfono sea defectuoso) dada una evidencia observada \( E \) (por ejemplo, el Puntaje del Sensor, la Línea de Ensamblaje y la Experiencia del Trabajador).

### Teorema de Bayes

- \( H \) es nuestra hipótesis sobre si un teléfono inteligente es **Defectuoso** o **No Defectuoso**.
- \( E \) son las características observadas (evidencia) del teléfono (por ejemplo, **Puntaje del Sensor**, **Línea de Ensamblaje**, y **Experiencia del Trabajador**).

#### Teorema de Bayes reescrito para teléfonos defectuosos:

$$
P(\text{Defectuoso} \mid E) = \frac{P(E \mid \text{Defectuoso}) \cdot P(\text{Defectuoso})}{P(E)}
$$

Sin embargo, como solo nos interesa comparar las **probabilidades relativas** de que un teléfono sea defectuoso o no defectuoso, podemos **ignorar \( P(E) \)**, ya que es una constante que aparece en ambos casos.

#### ¿Qué significa esto?

Significa que calculamos la probabilidad de cada clase (defectuoso, no defectuoso) y elegimos la que tenga el **mayor valor**. Por lo tanto, podemos **cancelar \( P(E) \)** en el Teorema de Bayes.

### Teorema de Bayes reescrito para Naïve Bayes:

$$
P(\text{Defectuoso} \mid E) \propto P(\text{Sensor Score} \mid \text{Defectuoso}) \cdot P(\text{Assembly Line} \mid \text{Defectuoso}) \cdot P(\text{Worker Experience} \mid \text{Defectuoso}) \cdot P(\text{Defectuoso})
$$

### ¿Por qué se llama “Naïve” Bayes?

El clasificador **Naïve Bayes** asume que todas las características son **independientes entre sí**, es decir, cada característica contribuye a la probabilidad de forma independiente.

Bajo esta suposición, podemos multiplicar las probabilidades condicionales individuales de cada característica:

#### Probabilidad condicional reescrita como producto:

$$
P(E \mid \text{Defectuoso}) = P(\text{Sensor Score} \mid \text{Defectuoso}) \cdot P(\text{Assembly Line} \mid \text{Defectuoso}) \cdot P(\text{Worker Experience} \mid \text{Defectuoso})
$$

Por supuesto, en la realidad las características **no siempre son independientes** (por ejemplo, la experiencia del trabajador podría influir en el puntaje del sensor). Sin embargo, esta suposición “ingenua” suele funcionar bastante bien en la práctica. Además, hace que el modelo sea **computacionalmente eficiente**, ya que transforma el Teorema de Bayes en un producto simple de probabilidades.

### Sustituyendo con los valores que hemos calculado:
$$
( P(\text{Sensor Score} \mid \text{Defectuoso}) = 0.4 )\\
( P(\text{Assembly Line} \mid \text{Defectuoso}) = 0.5 )\\
( P(\text{Worker Experience} \mid \text{Defectuoso}) = 0.25 )\\
( P(\text{Defectuoso}) = 0.2 )\\
$$

#### Cálculo:

$$
P(\text{Defectuoso} \mid E) \propto 0.4 \times 0.5 \times 0.25 \times 0.2 = 0.01
$$

Este es el valor proporcional de la probabilidad de que el teléfono sea **defectuoso**. Se comparará con el valor correspondiente para **no defectuoso** para tomar la decisión final.

### Realizando la clasificación final

Para realizar una clasificación final, también necesitamos calcular la probabilidad de que el teléfono **no sea defectuoso**, ya que debemos comparar las probabilidades de ambas clases (**defectuoso** y **no defectuoso**).

Así que... retomando nuestros cálculos de la sección 2, podemos calcular la probabilidad de **no defectuoso**:

> **Probabilidad no normalizada de no defectuoso** = 0.006875

### Comparación de probabilidades

Con las probabilidades de **defectuoso** y **no defectuoso**, podemos finalmente compararlas para hacer una clasificación:

- Probabilidad proporcional de **defectuoso** = 0.01
- Probabilidad proporcional de **no defectuoso** = 0.006875

Comparando:

$$
0.01 > 0.006875
$$

### Resultado final

En nuestro ejemplo, dado que \( 0.01 > 0.006875 \), el modelo **predice que el teléfono es Defectuoso**.



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = pd.read_csv("your_file.csv")

# Features (X) and Target (y)
X = data_encoded.drop('Defective', axis=1)
y = data_encoded['Defective']

# Split train & test
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", report)

### Normalización de las probabilidades

Hasta ahora, hemos trabajado con **probabilidades no normalizadas** (también llamadas **proporcionales**), que son suficientes para comparar y clasificar.  
Sin embargo, si queremos obtener **probabilidades reales** (es decir, que sumen 1 entre las dos clases), necesitamos **normalizar** los valores.

### ¿Cómo se normalizan?

La normalización consiste simplemente en dividir cada probabilidad proporcional entre la suma de ambas probabilidades.

La fórmula es:

$$
P(\text{Clase}) = \frac{\text{Probabilidad proporcional de la clase}}{\text{Suma de probabilidades proporcionales de todas las clases}}
$$

### Aplicándolo a nuestro ejemplo

Recordemos:

- Probabilidad proporcional de **Defectuoso** = 0.01
- Probabilidad proporcional de **No Defectuoso** = 0.006875

La suma de ambas es:

$$
0.01 + 0.006875 = 0.016875
$$

Entonces:

- Probabilidad real de **Defectuoso**:

$$
P(\text{Defectuoso}) = \frac{0.01}{0.016875} \approx 0.5926
$$

- Probabilidad real de **No Defectuoso**:

$$
P(\text{No Defectuoso}) = \frac{0.006875}{0.016875} \approx 0.4074
$$

### Interpretación

- Hay aproximadamente un **59,26%** de probabilidad de que el teléfono sea **Defectuoso**.
- Hay aproximadamente un **40,74%** de probabilidad de que el teléfono sea **No Defectuoso**.

Por lo tanto, **el modelo sigue prediciendo que el teléfono es Defectuoso**, pero ahora además conocemos las probabilidades reales asociadas a cada clase.
